In [9]:
import sys
import re

# tokens a reconhecer: comentarios, variais, strings, alfanumericos, delimitadores, limit
def analyser(inpt):
    recognized = []
    line = 1
    mo = re.finditer(r'((?P<COMMENT>\#.*)|(?P<SKIP>[ \t])|(?P<STRING>"[^"]*")|(?P<LIMIT>LIMIT\b)|(?P<NEWLINE>\n)|(?P<VAR>\?\w+)|(?P<IDENTIFIER>\@\w{2})|(?P<CONCEPT_NUM>\d+\b)|(?P<DELIM>[{}.])|(?P<CONCEPT_OTHER>[\w:]+)|(?P<ERROR>.))', inpt) #reconhecimento da gramatica
    for m in mo: # tokenizar
        d = m.groupdict()
        if d['COMMENT']: # '#'
            t = ("COMMENT", d['COMMENT'], line, m.span())
        elif d['SKIP']: # newlines, etc
            t = ("SKIP", d['SKIP'], line, m.span())
        elif d['STRING']: # sequencia de caracteres entre aspas
            t = ("STRING", d['STRING'], line, m.span())
        elif d['LIMIT']: # LIMIT literal
            t = ("LIMIT", d['LIMIT'], line, m.span())
        elif d['NEWLINE']:
            t = ("NEWLINE", d['NEWLINE'], line, m.span())
            line += 1
        elif d['VAR']: # '?' seguido de caracter alfanumerico
            t = ("VAR", d['VAR'], line, m.span())
        elif d['IDENTIFIER']: # identificador '@' seguido de caractere alfanumerico
            t = ("IDENTIFIER", d['IDENTIFIER'], line, m.span())
        elif d['CONCEPT_NUM']: # conceito numerico
            t = ("CONCEPT_NUM", d['CONCEPT_NUM'], line, m.span())
        elif d['DELIM']: # 'delimitadores {}'
            t = ("DELIM", d['DELIM'], line, m.span())
        elif d['CONCEPT_OTHER']: # conceito unicamente alfabetico
            t = ("CONCEPT_OTHER", d['CONCEPT_OTHER'], line, m.span())
        elif d['ERROR']: # o restante cai em erro
            t = ("ERROR", d['ERROR'], line, m.span())
        else:
            t = ("UNKNOWN", m.group(), line, m.span())

        if not d['SKIP'] and not d['COMMENT'] and not d['NEWLINE'] and t[0] != 'UNKNOWN': recognized.append(t)

    return recognized

example = ("""
    # DBPedia: obras de Chuck Berry)
    select ?nome ?desc where {
    ?s a dbo:MusicalArtist.
    ?s foaf:name "Chuck Berry"@en .
    ?w dbo:artist ?s.
    ?w foaf:name ?nome.
    ?w dbo:abstract ?desc
    } LIMIT 1000
    """
)

print(analyser(example))

print(analyser("123abc"))


for linha in sys.stdin:
    for tok in analyser(linha):
        print(tok)


[('CONCEPT_OTHER', 'select', 3, (42, 48)), ('VAR', '?nome', 3, (49, 54)), ('VAR', '?desc', 3, (55, 60)), ('CONCEPT_OTHER', 'where', 3, (61, 66)), ('DELIM', '{', 3, (67, 68)), ('VAR', '?s', 4, (73, 75)), ('CONCEPT_OTHER', 'a', 4, (76, 77)), ('CONCEPT_OTHER', 'dbo:MusicalArtist', 4, (78, 95)), ('DELIM', '.', 4, (95, 96)), ('VAR', '?s', 5, (101, 103)), ('CONCEPT_OTHER', 'foaf:name', 5, (104, 113)), ('STRING', '"Chuck Berry"', 5, (114, 127)), ('IDENTIFIER', '@en', 5, (127, 130)), ('DELIM', '.', 5, (131, 132)), ('VAR', '?w', 6, (137, 139)), ('CONCEPT_OTHER', 'dbo:artist', 6, (140, 150)), ('VAR', '?s', 6, (151, 153)), ('DELIM', '.', 6, (153, 154)), ('VAR', '?w', 7, (159, 161)), ('CONCEPT_OTHER', 'foaf:name', 7, (162, 171)), ('VAR', '?nome', 7, (172, 177)), ('DELIM', '.', 7, (177, 178)), ('VAR', '?w', 8, (183, 185)), ('CONCEPT_OTHER', 'dbo:abstract', 8, (186, 198)), ('VAR', '?desc', 8, (199, 204)), ('DELIM', '}', 9, (209, 210)), ('LIMIT', 'LIMIT', 9, (211, 216)), ('CONCEPT_NUM', '1000', 9, (2

KeyboardInterrupt: 